# 26. TensorFlow & Keras — `tensorflow · keras`
**Python Libraries Lab · Phase 2 · Deep Learning & AI**

_Build and train neural networks with a few readable lines._

✅ **Runs in Colab** as-is.

A neural network is a stack of layers. Each layer takes numbers in, multiplies them by weights, and passes the result on. During training the network compares its guesses with the right answers and nudges every weight to do a little better. TensorFlow is Google's engine for this, and Keras is its friendly interface, where a whole network is a short list of layers.

**In this notebook:** You will teach a network to read handwritten digits from zero to nine, using a small dataset that ships with scikit learn, so nothing has to download. You will watch the accuracy climb, spot overfitting on a chart, test it on digits it has never seen, and save the model to a file.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
TensorFlow is pre-installed in Colab. For faster training choose **Runtime → Change runtime type → T4 GPU** (optional; this notebook is small enough for CPU).

## Step 1 · Load the digits
1,797 small images of handwritten digits, already inside scikit-learn.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import numpy as np, matplotlib.pyplot as plt

digits = load_digits()                      # 1,797 handwritten digits, 8x8 pixels, built into scikit-learn
X = (digits.images / 16.0).astype("float32")  # scale pixels to 0..1
y = digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)

fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, label in zip(axes, X_train, y_train):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.show()

## Step 2 · Build the network
Flatten the 8x8 image to 64 numbers, two hidden layers, then 10 outputs (one per digit).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(0)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (CPU is fine here)")

model = keras.Sequential([
    keras.Input(shape=(8, 8)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.summary()

## Step 3 · Compile and train
20 epochs. `validation_split` keeps 20% of training data aside to watch for overfitting.

In [ ]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
print("Final training accuracy:  ", round(history.history["accuracy"][-1], 3))
print("Final validation accuracy:", round(history.history["val_accuracy"][-1], 3))

## Step 4 · Learning curves
If the two lines split apart, the model is memorising (overfitting).

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
a.plot(history.history["loss"], label="train"); a.plot(history.history["val_loss"], label="validation"); a.set_title("Loss (lower is better)"); a.legend()
b.plot(history.history["accuracy"], label="train"); b.plot(history.history["val_accuracy"], label="validation"); b.set_title("Accuracy"); b.legend()
plt.show()

## Step 5 · Test on unseen digits
The honest score. Then look at a few predictions.

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {acc:.3f}")

probs = model.predict(X_test[:8], verbose=0)
fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for ax, img, p, true in zip(axes, X_test[:8], probs, y_test[:8]):
    ax.imshow(img, cmap="gray_r"); ax.axis("off")
    ax.set_title(f"{p.argmax()} ({p.max():.0%})", color="green" if p.argmax() == true else "red", fontsize=9)
plt.show()

## Step 6 · Save and load
One `.keras` file holds the layers and the learned weights.

In [ ]:
model.save("digits.keras")
loaded = keras.models.load_model("digits.keras")
print("Loaded model predicts:", loaded.predict(X_test[:5], verbose=0).argmax(axis=1), "| true:", y_test[:5])

## ✍️ Your turn
Build a smaller model: `Flatten` then a single `Dense(10, activation='softmax')`. Compile it like above, train for 30 epochs (`verbose=0`), and store its **test accuracy** in `answer`. Tip: it should still score above 0.85.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    ok = answer is not None and 0.85 < float(answer) <= 1.0
    return ok, f"Expected an accuracy above 0.85, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
small = keras.Sequential([keras.Input(shape=(8, 8)), layers.Flatten(), layers.Dense(10, activation="softmax")])
small.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
small.fit(X_train, y_train, epochs=30, verbose=0)
answer = small.evaluate(X_test, y_test, verbose=0)[1]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Layer**
- **Sequential**
- **compile / fit**
- **Epoch**
- **Loss**
- **Overfitting**

**Next:** 27. PyTorch (`torch`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/27_pytorch.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install tensorflow scikit-learn matplotlib
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.